# 0. Environment Setup and Installations

In [ ]:
# 1. Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Install necessary libraries
print("Installing required libraries...")
!pip install -q transformers accelerate bitsandbytes torch torchvision Pillow mtcnn tqdm pandas scikit-learn matplotlib seaborn peft
print("Installations complete! Ready for Data Extraction.")

Mounted at /content/drive
Installing required libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.6 MB/s eta 0:00:00
Installations complete! Ready for Data Extraction.


# 1 Copy Img from Drive

In [ ]:
import os
import shutil
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

src_dir = '/content/drive/MyDrive/Deepfake_Project/Balanced_Dataset_V2_RetinaFace'
dest_dir = '/content/Balanced_Dataset_V2_RetinaFace'

print("Scanning files in Drive (this takes a minute)...")
files_to_copy = []

# Scanning the drive and building the folder structure at the destination
for root, dirs, files in os.walk(src_dir):
    dest_path = root.replace(src_dir, dest_dir)
    os.makedirs(dest_path, exist_ok=True)
    for file in files:
        files_to_copy.append(os.path.join(root, file))

print(f"Found {len(files_to_copy)} files. Starting parallel copy...")

# Copy function (skips already copied files in case of interruption)
def copy_file(file_path, max_retries=3):
    dest_path = file_path.replace(src_dir, dest_dir)
    if not os.path.exists(dest_path): #check if the file was already (in case of a crash)
        for attempt in range(max_retries):
            try:
                shutil.copy2(file_path, dest_path)
                return # successfull copy
            except FileNotFoundError as e:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt) #sleep between tries
                else:
                    print(f"\nFailed to copy {file_path} after {max_retries} attempts.")
                    raise e # crush only if the tries have played

# multi threading to copy files simultaneously
with ThreadPoolExecutor(max_workers=16) as executor:
    list(tqdm(executor.map(copy_file, files_to_copy), total=len(files_to_copy)))

print("Copy completed successfully!")

Scanning files in Drive (this takes a minute)...
Found 0 files. Starting parallel copy...


0it [00:00, ?it/s]

Copy completed successfully!


In [ ]:
import os
import zipfile
from tqdm.auto import tqdm

zip_path = '/content/drive/MyDrive/Deepfake_Project/Balanced_Dataset_V2_RetinaFace.zip'
dest_dir = '/content/Balanced_Dataset_V2_RetinaFace'

if os.path.exists(dest_dir) and len(os.listdir(dest_dir)) > 0:
    print("Dataset already extracted, skipping.")
else:
    print("Extracting dataset from zip...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        members = zip_ref.namelist()
        for member in tqdm(members, desc="Extracting"):
            zip_ref.extract(member, '/content')
    print(f"Extraction complete. Files at: {dest_dir}")

print(f"Contents: {os.listdir(dest_dir)}")

# 2. Model Initialization

In [ ]:
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Processor
processor = InstructBlipProcessor.from_pretrained("Salesforce/instructblip-vicuna-7b")

# Load Model in bfloat16 for stability and memory efficiency
model = InstructBlipForConditionalGeneration.from_pretrained(
    "Salesforce/instructblip-vicuna-7b",
    torch_dtype=torch.bfloat16
).to(device)

# --- FREEZING WEIGHTS (PEFT Strategy) ---
# We freeze the Vision Encoder and the LLM, and only train the Q-Former
for name, param in model.named_parameters():
    if "qformer" in name or "query_tokens" in name or "language_projection" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Verify freezing: print trainable parameters count
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/104k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Trainable parameters: 185,660,160 (2.35%)


# 3. Dataset Loading

In [ ]:
import os
import glob
import json
import random
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import multiprocessing

# Training Class: Performs concatenation of the answer so that the model learns
class TrainDeepfakeDataset(Dataset):
    def __init__(self, data_frame, processor):
        self.data = data_frame
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label_str = row['label']
        prompt = "is this photo real?"

        inputs = self.processor(images=image, text=prompt, return_tensors="pt")
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        prompt_len = inputs["input_ids"].shape[0]

        answer_tokens = self.processor.tokenizer(" " + label_str, return_tensors="pt", add_special_tokens=False).input_ids.squeeze(0)
        answer_len = answer_tokens.shape[0]

        inputs["input_ids"] = torch.cat([inputs["input_ids"], answer_tokens])
        inputs["attention_mask"] = torch.cat([inputs["attention_mask"], torch.ones(answer_len, dtype=torch.long)])

        prompt_labels = torch.full((prompt_len,), -100, dtype=torch.long)
        inputs["labels"] = torch.cat([prompt_labels, answer_tokens])
        return inputs

# --- data paths ---
base_dir = '/content/Balanced_Dataset_V2_RetinaFace'
fake_images_folder = os.path.join(base_dir, 'Fake')
real_images_folder = os.path.join(base_dir, 'Real')

all_paths = []
all_labels = []

def collect_images(folder, label, target_paths, target_labels):
    if os.path.exists(folder):
        paths = glob.glob(os.path.join(folder, '**', '*.*'), recursive=True)
        valid_paths = sorted([p for p in paths if p.lower().endswith(('.png', '.jpg', '.jpeg'))])
        target_paths.extend(valid_paths)
        target_labels.extend([label] * len(valid_paths))

collect_images(fake_images_folder, "No", all_paths, all_labels)
collect_images(real_images_folder, "Yes", all_paths, all_labels)

print(f"Total images found: {len(all_paths)}")

if len(all_paths) > 0:
    df = pd.DataFrame({'image_path': all_paths, 'label': all_labels})

    # Extract video_id from path (the parent folder name)
    df['video_id'] = df['image_path'].apply(lambda p: os.path.basename(os.path.dirname(p)))

    # --- טען את מזהי ה-Train שנקבעו בסקריפט החילוץ ---
    train_ids_path = '/content/drive/MyDrive/Deepfake_Project/train_video_ids.json'
    with open(train_ids_path, 'r') as f:
        train_video_ids = set(json.load(f))

    print(f"Loaded {len(train_video_ids)} train video IDs from JSON")

    train_df = df[df['video_id'].isin(train_video_ids)].reset_index(drop=True)
    print(f"Training videos matched: {train_df['video_id'].nunique()} | Training images: {len(train_df)}")

    train_dataset = TrainDeepfakeDataset(train_df, processor)

    optimal_workers = min(8, multiprocessing.cpu_count())
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=4,
        shuffle=True,
        num_workers=optimal_workers,
        pin_memory=True,
        prefetch_factor=2
    )
    print("Train DataLoader ready!")
else:
    print("ERROR: No images loaded.")

Total images found: 19904
Training images: 1990
Train paths saved to: /content/drive/MyDrive/Deepfake_Project/train_paths.csv
Train DataLoader ready!


# 4. Fine tuning Training Loop

In [ ]:
import torch
import os
from torch.optim import AdamW
from tqdm import tqdm
import json

# --- 1. Optimizer Setup ---
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.05,
    betas=(0.9, 0.999)
)

# --- 2. Training Loop ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

print("Starting Fine-Tuning Quality Pass...")
training_loss_history = []
loop = tqdm(train_dataloader, leave=True, desc="Training")

for batch in loop:
    optimizer.zero_grad()

    batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

    outputs = model(**batch)
    loss = outputs.loss

    loss.backward()

    torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)

    optimizer.step()

    current_loss = loss.item()
    training_loss_history.append(current_loss)
    loop.set_postfix(loss=f"{current_loss:.4f}")

# --- 3. Save the Model ---
save_dir = "/content/drive/MyDrive/Deepfake_Project/V2_Weights"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "qformer_finetuned_quality.pth")

torch.save(model.qformer.state_dict(), save_path)
print(f"\nSuccess: Fine-tuned Q-Former weights saved to {save_path}")

# --- 4. save loss history ---

with open(os.path.join(save_dir, "loss_history.json"), "w") as f:
    json.dump(training_loss_history, f)

Starting Fine-Tuning Quality Pass...


Training: 100%|██████████| 498/498 [01:45<00:00,  4.71it/s, loss=1.5278]



Success: Fine-tuned Q-Former weights saved to /content/drive/MyDrive/Deepfake_Project/V2_Weights/qformer_finetuned_quality.pth
